In [0]:
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("base_schema_name", "")
dbutils.widgets.text("nlc_schema_name", "")

In [0]:
catalog_name=dbutils.widgets.get('catalog_name')
base_schema_name=dbutils.widgets.get('base_schema_name')
schema_name=dbutils.widgets.get('nlc_schema_name')

In [0]:
tables = ['catalog_page', 'catalog_sales', 'customer_address', 'customer_demographics', 'date_dim', 'item', 'promotion', 'ship_mode', 'store', 'store_sales']

for table in tables:
    nlc_table_name = f"{catalog_name}.{schema_name}.{table}"
    base_table_name = f"{catalog_name}.{base_schema_name}.{table}"
    spark.sql(f"CREATE TABLE {nlc_table_name} as select * from {base_table_name}")

In [0]:
# Add Primary Key Constraints and Rely Clause on Dim Tables 
spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.date_dim
ALTER COLUMN d_date_sk_1 SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.date_dim
ADD CONSTRAINT pk_date_dim PRIMARY KEY (d_date_sk_1) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_address
ALTER COLUMN ca_address_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_address
ADD CONSTRAINT pk_customer_address PRIMARY KEY (ca_address_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.promotion
ALTER COLUMN p_promo_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.promotion
ADD CONSTRAINT pk_promotion PRIMARY KEY (p_promo_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.item
ALTER COLUMN i_item_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.item
ADD CONSTRAINT pk_item PRIMARY KEY (i_item_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.catalog_page
ALTER COLUMN cp_catalog_page_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.catalog_page
ADD CONSTRAINT pk_cp PRIMARY KEY (cp_catalog_page_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_demographics
ALTER COLUMN cd_demo_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_demographics
ADD CONSTRAINT pk_cd PRIMARY KEY (cd_demo_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.ship_mode
ALTER COLUMN sm_ship_mode_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.ship_mode
ADD CONSTRAINT pk_sm PRIMARY KEY (sm_ship_mode_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.store
ALTER COLUMN s_store_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.store
ADD CONSTRAINT pk_s PRIMARY KEY (s_store_sk) RELY
""")

In [0]:
# Z-order
spark.sql(f"""
    OPTIMIZE {catalog_name}.{schema_name}.store_sales 
    ZORDER BY (ss_sold_date_sk, ss_addr_sk)
""")

spark.sql(f"""
    OPTIMIZE {catalog_name}.{schema_name}.catalog_sales 
    ZORDER BY (cs_sold_date_sk, cs_bill_addr_sk)
""")

In [0]:
# Run VACUUM and ANALYZE on Fact Tables
fact_tables = ['catalog_sales', 'store_sales']

for fact_table in fact_tables:
    full_fact_table_name = f"{catalog_name}.{schema_name}.{fact_table}"
    spark.sql(f"VACUUM {full_fact_table_name}")
    spark.sql(f"ANALYZE TABLE {full_fact_table_name} COMPUTE STATISTICS")



In [0]:
# Run OPTIMIZE, VACUUM and ANALYZE on Dim Tables
dim_tables = ['catalog_page', 'customer_address', 'customer_demographics', 'date_dim', 'item', 'promotion', 'ship_mode', 'store']

for dim_table in dim_tables:
    full_dim_table_name = f"{catalog_name}.{schema_name}.{dim_table}"
    spark.sql(f"OPTIMIZE {full_dim_table_name}")
    spark.sql(f"VACUUM {full_dim_table_name}")
    spark.sql(f"ANALYZE TABLE {full_dim_table_name} COMPUTE STATISTICS")